<!-- SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved. -->
<!-- SPDX-License-Identifier: Apache-2.0 -->

# Alpamayo 2 Edge: Navigation-Conditioned Inference

Alpamayo 2 Edge is a 2.65B-parameter driving model built on the Cosmos3-Edge vision-language backbone, with a trajectory diffusion expert. Given six camera views, the vehicle's recent motion, and a natural-language navigation instruction such as *"Turn right in 30m"*, it writes its driving reasoning and predicts 64 future poses at 0.1-second intervals (6.4 seconds).

This notebook runs the example in `examples/nav_samples.json` end to end:
1. **Load a clip** from the NVIDIA [PhysicalAI-AV dataset](https://huggingface.co/datasets/nvidia/PhysicalAI-Autonomous-Vehicles) and show the 22 camera frames the model reads.
2. **Run the model** to sample six reasoning continuations, each with its own trajectory.
3. **Plot the trajectories** in a bird's-eye view.

### Requirements

- A CUDA GPU. The BF16 weights alone take about 5.3 GB of GPU memory; on an H100 the six-candidate example below peaked at about 9.2 GB, and the inference cell prints its own peak.
- A Hugging Face account with access to [nvidia/Alpamayo2-Edge](https://huggingface.co/nvidia/Alpamayo2-Edge) and to the PhysicalAI-AV dataset. Accept the dataset's access terms on Hugging Face before running the notebook.
- This repository installed with the optional `dataset` extra, which provides the PhysicalAI-AV SDK, plus Jupyter and matplotlib, which are not part of the locked dependencies.

From the repository root:

```bash
uv sync --locked --extra dataset --dev
source .venv/bin/activate
hf auth login
uv run --locked --extra dataset --with jupyter --with matplotlib jupyter lab notebooks/inference.ipynb
```

In [ ]:
%matplotlib inline
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import physical_ai_av
import torch

from alpamayo2_edge.chat_template import CAMERA_DISPLAY_NAMES
from alpamayo2_edge.checkpoint import load_model
from alpamayo2_edge.helper import prepare_model_inputs, to_device
from alpamayo2_edge.inference import load_clip_manifest, predict
from alpamayo2_edge.load_physical_aiavdataset import load_physical_aiavdataset

In [ ]:
# A Hugging Face repository ID at the revision pinned in the README, or a local model directory.
MODEL_ID = "nvidia/Alpamayo2-Edge"
MODEL_REVISION = "f8951d095de3589b3face1dec3f4394c8ecedb38"
# A pinned PhysicalAI-AV dataset commit; None selects the latest commit.
DATASET_REVISION = "2ac2e9c056e0c8426face5bfad64b0961a295b28"
DEVICE = "cuda"
DTYPE = "bfloat16"
# PyTorch SDPA is the default; "flash_attention_2" also works when flash-attn is installed.
ATTN_IMPLEMENTATION = "sdpa"

if not torch.cuda.is_available():
    raise RuntimeError("This notebook needs a CUDA GPU.")

## Load the example clip

`examples/nav_samples.json` selects clip `ea7bbd31-b7a5-4972-8dbd-7089e6b53de4` at t0 = 4.0 s, as the vehicle approaches an intersection, together with the navigation instruction *"Turn right in 30m"*. The dataset does not include route instructions, so each example supplies its own.

`load_physical_aiavdataset` streams the model inputs from Hugging Face through the PhysicalAI-AV SDK:
- **22 camera frames from six cameras**: five frames (−2, −1, −0.5, −0.1 and 0 s) from the front and front telephoto cameras, and three (−1, −0.1 and 0 s) from each of the other four.
- **20 past ego poses** from −1.9 s to t0 at 0.1 s intervals, in the ego frame at t0 (x forward, y left, z up).

It does not load future poses or camera calibration. Passing your own SDK interface as `avdi` pins the dataset revision and lets several clips reuse it.

In [ ]:
# Jupyter starts the kernel in notebooks/; running from the repository root also works.
candidates = [Path("examples/nav_samples.json"), Path("../examples/nav_samples.json")]
manifest = next((path for path in candidates if path.is_file()), None)
if manifest is None:
    raise FileNotFoundError("Keep this notebook in the repository's notebooks/ directory.")
example = load_clip_manifest(manifest, index=0)  # clip_id, t0_us and nav_text

avdi = physical_ai_av.PhysicalAIAVDatasetInterface(revision=DATASET_REVISION)
data = load_physical_aiavdataset(**example, avdi=avdi)

print(f"Clip {example['clip_id']} at t0 = {example['t0_us'] / 1e6:.1f} s")
print(f"Navigation instruction: {example['nav_text']!r}")
print(f"Dataset revision: {data['provenance'].get('dataset_revision')}")
for key in ("image_frames", "camera_indices", "absolute_timestamps", "ego_history_xyz"):
    print(f"  {key:20s} {str(data[key].dtype):14s} {tuple(data[key].shape)}")

### Camera inputs

Each row is one camera, in the order the model reads them, and each column is one sampling time. Panel titles give the actual capture time relative to t0, which can differ slightly from the requested time.

In [ ]:
cameras = data["camera_indices"].tolist()
seconds = (data["absolute_timestamps"].reshape(-1) - example["t0_us"]).numpy() / 1e6
offsets = np.array([-2.0, -1.0, -0.5, -0.1, 0.0])  # requested sampling times in seconds

fig, axes = plt.subplots(len(CAMERA_DISPLAY_NAMES), len(offsets), figsize=(16, 11))
for ax in axes.flat:
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_frame_on(False)
for row, (camera, name) in enumerate(CAMERA_DISPLAY_NAMES.items()):
    axes[row, 0].set_ylabel(name, rotation=0, ha="right", va="center")
    for index in [i for i, c in enumerate(cameras) if c == camera]:
        ax = axes[row, np.abs(offsets - seconds[index]).argmin()]
        ax.imshow(data["image_frames"][index, 0].permute(1, 2, 0).numpy())
        ax.set_title(f"{seconds[index]:+.2f} s", fontsize=9)
fig.suptitle(f"Camera inputs, navigation instruction: {example['nav_text']!r}")
fig.tight_layout()
plt.show()

## Load the model and run inference

`load_model` downloads the checkpoint at the pinned revision (about 5.3 GB) and rejects incomplete or mismatched weights. `prepare_model_inputs` then builds the prompt: each camera's name, a relative time label before each of the 22 images, the images resized to their per-camera pixel budgets, placeholders that the model fills with ego history tokens, and the instruction as plain text (`Navigation input is: Turn right in 30m`). The prompt ends with an empty `<think></think>` block, so the model writes its driving reasoning directly.

`predict` is the function the command-line tool runs: it seeds token sampling and diffusion noise, then calls `model.sample_trajectories_from_data` under `torch.inference_mode()` and BF16 autocast. The settings are the command-line defaults except for the number of candidates: `num_traj_sets=6` and `num_traj_samples=1` give six candidates, each an independently sampled reasoning continuation followed by one trajectory from 10 diffusion steps. Sampled text and trajectories can change with the GPU, attention backend and library versions.

In [ ]:
model, _ = load_model(
    MODEL_ID,
    device=DEVICE,
    dtype=DTYPE,
    attn_implementation=ATTN_IMPLEMENTATION,
    revision=MODEL_REVISION,  # used only for Hugging Face repository IDs
)
print(f"{sum(p.numel() for p in model.parameters()) / 1e9:.2f}B parameters on {model.device}")

In [ ]:
inputs = to_device(prepare_model_inputs(data, model.config, model.tokenizer), device=DEVICE)
print("Prompt length:", inputs["tokenized_data"]["input_ids"].shape[1], "tokens")

SETTINGS = dict(
    seed=43,
    top_p=0.98,
    top_k=None,
    temperature=0.6,
    num_traj_sets=6,  # 6 x 1 = six candidates, each with
    num_traj_samples=1,  # its own reasoning and trajectory
    max_generation_length=256,
    diffusion_kwargs={"inference_step": 10},
)
torch.cuda.reset_peak_memory_stats(DEVICE)
pred_xyz, pred_rot, extra = predict(model, inputs, SETTINGS, DEVICE)
print("pred_xyz:", tuple(pred_xyz.shape), " pred_rot:", tuple(pred_rot.shape))
print(f"Peak GPU memory allocated: {torch.cuda.max_memory_allocated(DEVICE) / 1e9:.1f} GB")

# [candidates, 64, 3] in the ego frame at t0; trajectories[:, i] is at t0 + 0.1 * (i + 1) s.
trajectories = pred_xyz[0].reshape(-1, 64, 3).float().cpu().numpy()
for k, (xyz, cot) in enumerate(zip(trajectories, extra["cot"].reshape(-1)), start=1):
    forward, left = xyz[-1, :2]
    side = "left" if left >= 0 else "right"
    print(f"\n[{k}] {cot}")
    print(f"    after 6.4 s: {forward:.1f} m forward, {abs(left):.1f} m {side}")

## Bird's-eye view

The trajectories are in the ego frame at t0 (x forward, y left), drawn with forward pointing up and left on the left. The numbers in the legend match the reasoning printed above. For reference, the dashed line is the path the vehicle actually drove, read separately from the dataset's egomotion; the model does not see it.

In [ ]:
history = data["ego_history_xyz"][0].numpy()  # [20, 3] from -1.9 s to t0

# The logged future path for reference; it is not a model input.
egomotion = avdi.get_clip_feature(
    example["clip_id"], avdi.features.LABELS.EGOMOTION, maybe_stream=True
)
times = example["t0_us"] + np.arange(0, 6_400_001, 100_000)  # t0, then 64 steps of 0.1 s
logged = None
if times[-1] <= egomotion.time_range[1]:
    poses = egomotion(times).pose
    positions = np.asarray(poses.translation)
    logged = poses.rotation[0].inv().apply(positions - positions[0])[1:]

PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]
fig, ax = plt.subplots(figsize=(4.5, 8))
ax.plot(
    history[:, 1], history[:, 0], "-.", color="#52514e", lw=2, label="Ego history (-1.9 s to t0)"
)
for k, xyz in enumerate(trajectories):
    labeled = k < len(PALETTE)  # candidates beyond six are drawn in gray
    ax.plot(
        xyz[:, 1],
        xyz[:, 0],
        color=PALETTE[k] if labeled else "#898781",
        lw=2,
        label=f"Candidate {k + 1}" if labeled else None,
    )
if logged is not None:
    ax.plot(logged[:, 1], logged[:, 0], "--", color="#0b0b0b", lw=2, label="Logged future path")
ax.plot(0, 0, "^", color="#0b0b0b", ms=10, label="Ego vehicle at t0")
ax.invert_xaxis()  # positive y (left) on the left
ax.set_aspect("equal", adjustable="datalim")
ax.set_xlabel("y (m), positive to the left")
ax.set_ylabel("x (m), forward")
ax.set_title(f"Navigation: {example['nav_text']!r}")
ax.grid(color="#e1e0d9", lw=0.8)
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1), fontsize=9)
plt.show()

## Optional: accelerated inference

`alpamayo2_edge.accel` runs the prompt prefill, reasoning decoding and the diffusion expert with `torch.compile` and CUDA graphs; this is the Python equivalent of `--accel tuned`. With six candidates, as here, the prefill runs eagerly and the expert graph replays only when all six continuations have the same length, so the speedup comes mainly from decoding. Acceleration keeps the model, BF16 precision and sampling semantics, but needs a BF16 model and FlashAttention 2 (see `docs/acceleration.md`); without them, the cell below only prints a message. The first accelerated call compiles and captures the graphs, which can take a minute or more and prints autotuning logs; later calls with the same shapes replay them. Accelerated outputs are not bitwise identical to the default path, so a sampled token can occasionally differ.

In [ ]:
import time

from alpamayo2_edge.accel import AccelConfig, enable_acceleration, resolve_kernels


def timed_predict():
    torch.cuda.synchronize(DEVICE)
    start = time.perf_counter()
    outputs = predict(model, inputs, SETTINGS, DEVICE)
    torch.cuda.synchronize(DEVICE)
    return outputs, time.perf_counter() - start


if DTYPE != "bfloat16":
    print("The accelerated path needs a BF16 model; skipping it.")
elif resolve_kernels("auto", DEVICE)["decode"] is None:
    print("FlashAttention 2 (flash-attn) is not installed; skipping the accelerated path.")
else:
    _, default_seconds = timed_predict()
    accel = enable_acceleration(model, AccelConfig(mode="tuned"))
    try:
        timed_predict()  # warm-up: compiles and captures the CUDA graphs
        (_, _, fast_extra), fast_seconds = timed_predict()
    finally:
        accel.remove()  # restore the default inference path
    print(f"Model call: {default_seconds:.2f} s default, {fast_seconds:.2f} s accelerated")
    for k, cot in enumerate(fast_extra["cot"].reshape(-1), start=1):
        print(f"[{k}] {cot}")